# Dra. Estadística IA — Notebook para Google Colab

Versión de trabajo para aprender estadística con una profesora conversacional, recuperar notas didácticas básicas y ejecutar cálculos reales con Python.

**Cómo usarlo:** ejecuta las celdas en orden. La clave se solicitará de forma privada; no la escribas dentro del notebook.

## 1. Instalar dependencias

In [ ]:
!pip -q install requests numpy pandas

## 2. Configurar OpenRouter

In [ ]:
import os, re, math, getpass, requests
import numpy as np
import pandas as pd

OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"
OPENROUTER_CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"

api_key = getpass.getpass("Pega tu clave API de OpenRouter (entrada oculta): ").strip()
if not api_key:
    raise ValueError("Se necesita una clave de OpenRouter para activar el chat.")

## 3. Descubrir modelos gratuitos compatibles con texto

In [ ]:
def get_free_chat_models():
    response = requests.get(OPENROUTER_MODELS_URL, timeout=25)
    response.raise_for_status()
    free = []
    for model in response.json().get("data", []):
        pricing = model.get("pricing") or {}
        try:
            p_in = float(pricing.get("prompt", "inf"))
            p_out = float(pricing.get("completion", "inf"))
        except (TypeError, ValueError):
            continue
        architecture = model.get("architecture") or {}
        modality = str(architecture.get("modality") or "").lower()
        if p_in == 0 and p_out == 0 and (not modality or "text" in modality):
            free.append({"id": model.get("id"), "name": model.get("name") or model.get("id")})
    return sorted([m for m in free if m["id"]], key=lambda x: x["name"].lower())

models = get_free_chat_models()
print(f"Modelos gratuitos de texto detectados: {len(models)}")
for i, model in enumerate(models[:50], 1):
    print(f"{i:02d}. {model['name']} — {model['id']}")

## 4. System Prompt pedagógico

In [ ]:
SYSTEM_PROMPT = """
Eres Dra. Estadística IA, profesora cercana, paciente, rigurosa y didáctica.
Enseña progresivamente: teoría breve seguida de práctica; comprensión lógica antes de fórmulas;
ejemplos cotidianos; explicación por etapas; prevención de errores; resúmenes; ejercicios autónomos.
Aclara notaciones y convenciones. Distingue posición y valor en cuantiles.
En varianza y desviación estándar distingue población y muestra, denominadores y unidades.
No inventes resultados, datos, fuentes ni citas. Explica supuestos y límites.
Adapta la profundidad al estudiante y corrige con respeto y motivación.
"""
print("System Prompt cargado.")

## 5. RAG inicial modular

In [ ]:
KNOWLEDGE_BASE = [
    {"topic":"Conceptos y fundamentos",
     "text":"La estadística recolecta, organiza, resume, analiza e interpreta datos. Población es el conjunto completo; muestra es una parte. Parámetro describe población y estadístico describe muestra. Las variables pueden ser cualitativas o cuantitativas."},
    {"topic":"Estadística descriptiva",
     "text":"La estadística descriptiva resume los datos observados con tablas, gráficos y medidas. Antes de analizar, revisa unidades, valores faltantes y errores."},
    {"topic":"Tendencia central y dispersión",
     "text":"La media es suma entre número de datos; mediana es el valor central ordenado; moda es el más frecuente. Rango es máximo menos mínimo. Varianza poblacional divide entre N y varianza muestral insesgada entre n-1."},
    {"topic":"Cuartiles, deciles y percentiles",
     "text":"Los cuantiles dividen datos ordenados en partes. Distingue la posición calculada del valor que ocupa esa posición. Los métodos pueden variar; indica la convención usada."},
    {"topic":"Ejercicios y procedimientos",
     "text":"Identifica qué se pregunta, reconoce variable y unidades, ordena datos, elige método, calcula, interpreta en contexto y comprueba si el resultado tiene sentido."}
]

def retrieve_context(question, top_k=3):
    tokens = set(re.findall(r"[a-záéíóúñ0-9]+", question.lower()))
    ranked = []
    for doc in KNOWLEDGE_BASE:
        doc_tokens = set(re.findall(r"[a-záéíóúñ0-9]+", (doc["topic"]+" "+doc["text"]).lower()))
        score = len(tokens & doc_tokens)
        if score:
            ranked.append((score, doc))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

print("Bloques RAG:", [d["topic"] for d in KNOWLEDGE_BASE])

## 6. Chat con fallback automático

In [ ]:
def ask_dra(question, selected_model=None):
    context_docs = retrieve_context(question)
    context = "\n\n".join(f"[{d['topic']}] {d['text']}" for d in context_docs)
    messages = [
        {"role":"system","content":SYSTEM_PROMPT + "\n\nCONTEXTO RAG:\n" + (context or "Sin nota recuperada.")},
        {"role":"user","content":question}
    ]
    candidates = []
    if selected_model:
        candidates.append(selected_model)
    candidates.extend([m["id"] for m in models if m["id"] != selected_model])
    headers = {"Authorization":f"Bearer {api_key}", "Content-Type":"application/json"}
    errors = []
    for model_id in candidates:
        try:
            r = requests.post(OPENROUTER_CHAT_URL, headers=headers,
                              json={"model":model_id,"messages":messages,"temperature":0.4},
                              timeout=90)
            if r.ok:
                answer = r.json()["choices"][0]["message"].get("content","")
                if answer:
                    return answer, model_id
            errors.append(f"{model_id}: HTTP {r.status_code} {r.text[:120]}")
        except Exception as e:
            errors.append(f"{model_id}: {e}")
    raise RuntimeError("Fallaron los modelos probados: " + " | ".join(errors[:5]))

if not models:
    print("No hay modelos gratuitos de texto detectados ahora. Reintenta más tarde.")
else:
    model_ids = [m["id"] for m in models]
    print("Modelo inicial sugerido:", models[0]["name"], models[0]["id"])

## 7. Pregunta a Dra. Estadística IA

In [ ]:
pregunta = "Explícame con un ejemplo cotidiano la diferencia entre varianza poblacional y muestral."
respuesta, modelo_usado = ask_dra(pregunta, selected_model=models[0]["id"])
print("Modelo utilizado:", modelo_usado)
print("\n", respuesta)

## 8. Motor estadístico real con Python

In [ ]:
def calcular_estadistica(datos):
    x = np.asarray(datos, dtype=float)
    if x.size == 0 or not np.isfinite(x).all():
        raise ValueError("Introduce datos numéricos finitos.")
    modas, frecuencias = np.unique(x, return_counts=True)
    max_freq = frecuencias.max()
    moda = modas[frecuencias == max_freq].tolist() if max_freq > 1 else []
    return {
        "n": int(x.size),
        "media": float(np.mean(x)),
        "mediana": float(np.median(x)),
        "moda": moda,
        "mínimo": float(np.min(x)),
        "máximo": float(np.max(x)),
        "rango": float(np.ptp(x)),
        "varianza_poblacional": float(np.var(x, ddof=0)),
        "desviación_poblacional": float(np.std(x, ddof=0)),
        "varianza_muestral": float(np.var(x, ddof=1)) if x.size > 1 else None,
        "desviación_muestral": float(np.std(x, ddof=1)) if x.size > 1 else None,
        "cuartiles_método_lineal": np.quantile(x, [0.25, 0.50, 0.75], method="linear").tolist()
    }

datos = [8, 10, 10, 12, 15, 17, 18]
resultado = calcular_estadistica(datos)
pd.Series(resultado, name="Resultado")

## 9. Prueba con tus propios datos

In [ ]:
entrada = input("Escribe datos separados por coma (usa punto decimal): ")
datos_usuario = [float(v.strip()) for v in entrada.split(",") if v.strip()]
resultado_usuario = calcular_estadistica(datos_usuario)
display(pd.DataFrame({"Medida": list(resultado_usuario.keys()),
                      "Resultado": [str(v) for v in resultado_usuario.values()]}))

## Alcance de esta versión

- El catálogo de OpenRouter es dinámico y puede cambiar.
- El fallback prueba modelos gratuitos de texto detectados; no garantiza que todos estén operativos en cada momento.
- El RAG es una colección pequeña de notas didácticas iniciales, no una biblioteca académica con citas verificadas.
- Python ejecuta los cálculos en las celdas; el modelo conversacional no ejecuta automáticamente código.
- Los cuartiles se calculan con el método lineal de NumPy. Otras convenciones pueden producir valores diferentes.